# C5-neural-networks — Review

Work through this notebook *after* the three lesson sessions and (ideally)
the practice sets.
It is a consolidation tool: a concept summary table, the
formula-and-contract sheet, a 14-item self-quiz spanning every taught
concept, and pointers on what to redo.
Quiz answers are collapsed at the very end — commit to your answers before
looking.

In [ ]:
import numpy as np

## Concept summary

| Concept | One-line summary | Key fact to retain |
|---|---|---|
| Perceptron | Weighted sum then threshold: $z = \sum_k w_k x_k + b$, output $1$ iff $z \ge 0$ | Boundary fires ($z = 0 \to 1$); scaling $(w, b)$ by a positive constant changes no output |
| Threshold (step) activation | $\mathrm{step}(z) = \mathbb{1}[z \ge 0]$ as floats, elementwise | `(z >= 0).astype(float)` — `>` instead of `>=` silently flips every boundary point |
| Activation functions | Fixed elementwise nonlinearities between affine layers | Without one, a stack of affine layers collapses to a single affine layer ($W^{\mathrm{eff}}, b^{\mathrm{eff}}$) |
| ReLU | $\mathrm{relu}(z) = \max(z, 0)$: one-sided identity, kink at 0 | Sums of scaled shifted ReLUs = piecewise-linear functions; slopes add past each shift |
| MLP architecture | Layers of units composed: $d_{\text{in}} \to h \to d_{\text{out}}$; $W$ is $(d_{\text{out}}, d_{\text{in}})$, **row = unit** | Forward pass = `affine_layer` and activation, alternating; shapes checked before running |
| Decision boundaries | Step unit = half-plane detector ($w$ points into the firing side); gates AND/OR them | Convex polygon = AND of edge detectors, bias $-(m - 0.5)$; union of pieces needs a second layer + OR |
| Weight-init variance | Random weights: $\operatorname{Var}[z] = C\sigma^2$ for $C$ standardized independent inputs | Stability demands $\sigma = 1/\sqrt{C}$; per-layer spread factor is $\sigma\sqrt{C}$ |

## Formula and contract sheet

**The pinned helpers (C6-pytorch rebuilds these by name):**

```python
def affine_layer(x, W, b):
    # x: (n, d_in)  W: (d_out, d_in)  b: (d_out,)  ->  (n, d_out)
    # z[i, j] = sum_k x[i, k] * W[j, k] + b[j]
    return (x[:, None, :] * W[None, :, :]).sum(axis=2) + b

def step_activation(z):
    return (z >= 0).astype(float)
```

**Activations:** $\mathrm{step}(0) = 1$; $\tanh$ range $(-1, 1)$,
derivative $1 - \tanh^2$; $\mathrm{relu}(z) = \max(z, 0)$ via
`np.maximum(z, 0.0)` (never `np.max`).

**Linear collapse:** two activation-free affine layers equal one:
$W^{\mathrm{eff}}_{ik} = \sum_j W^{(2)}_{ij} W^{(1)}_{jk}$,
$\;b^{\mathrm{eff}}_i = \sum_j W^{(2)}_{ij} b^{(1)}_j + b^{(2)}_i$.

**Detector design:** rewrite the target inequality as
$w \cdot x + b \ge 0$, read off $(w, b)$; $w$ is normal to the boundary
and points into the firing side.

**Gates on $m$ detectors (weights all 1):** AND bias $-(m - 0.5)$;
OR bias $-0.5$; at-least-$r$ bias $-(r - 0.5)$; NOT = weight $-1$,
bias $0.5$.
Half-integer thresholds because the firing count is an integer.

**Init variance chain (Session 3):** group-form independence fact →
products $W_k x_k$ mutually independent;
$\operatorname{Var}[W_k x_k] = E[W_k^2]E[x_k^2] = \sigma^2$ (means zero) →
$\operatorname{Var}[z] = C \sigma^2$ → set $= 1$ →
$\boxed{\sigma = 1/\sqrt{C}}$.
Per-layer spread factor $\sigma \sqrt{C}$; inputs of variance $v$ give
$\operatorname{Var}[z] = C\sigma^2 v$; weight mean $\mu \ne 0$ gives
$C(\mu^2 + \sigma^2)$.

## Self-quiz (14 items)

Answer everything before opening the collapsed answers at the very end.

**Q1.** A perceptron has $w = (1, -2)$, $b = 3$. Compute $z$ and the
output at $x = (2, 2)$.

**Q2.** Design $(w, b)$ so a perceptron fires exactly when
$x_2 \le x_1 - 1$.

**Q3.** Evaluate the step activation at $z = -0.5$, $0$, and $0.5$, and
state the boundary convention in words.

**Q4.** `(z > 0)` and `(z >= 0)`: on exactly which inputs do they differ,
and which does this course pin? Why does the difference bite constantly
in designed-weight problems?

**Q5.** Give the ranges of step, tanh, and ReLU. Which of the three is
differentiable everywhere?

**Q6.** In one sentence: what goes wrong if the hidden activation is
removed from a $2 \to 4 \to 1$ network, and what are the effective
parameters called?

**Q7.** $r(z) = 3\,\mathrm{relu}(z) - 3\,\mathrm{relu}(z - 2)$: evaluate
at $z = -1, 1, 5$ and describe $r$ piece by piece.

**Q8.** A "ReLU" implemented as `np.max(z, 0)` returns a scalar. What
happened, and what is the fix?

**Q9.** For a $4 \to 6 \to 2$ MLP: give the shapes of $W^{(1)}, b^{(1)},
W^{(2)}, b^{(2)}$, and remember which axis of $W$ enumerates units.

**Q10.** Forward $x = (1, 1)$ by hand through: hidden
$W^{(1)} = \begin{pmatrix} 1 & -1 \\ 0 & 2 \end{pmatrix}$,
$b^{(1)} = (0, -1)$, step activation; output affine $w^{(2)} = (2, -1)$,
$b^{(2)} = -0.5$. Give $z^{(1)}$, $h$, and $y$.

**Q11.** Write the detector $(w, b)$ for the half-plane
$x_1 + 2 x_2 \le 8$, and say which way $w$ points relative to the
boundary line.

**Q12.** For polygon membership by AND-ing $m = 4$ edge detectors: give
the output unit's weights and bias, and explain why the polygon's edges
end up *inside*.

**Q13.** Width $C = 49$: what $\sigma$ does the init rule prescribe?
And with $\sigma = 0.2$ at that width, what per-layer spread factor
results — growing or shrinking?

**Q14.** $W \perp x$, both mean 0, $\operatorname{sd}[W] = 0.3$,
$\operatorname{Var}[x] = 1$: compute $\operatorname{Var}[Wx]$ and name
the two facts the computation leans on.

## What to redo, per weak spot

| If this felt shaky | Redo these |
|---|---|
| Perceptron evaluation / design | p01, p05, p20; Session 1 §1–2 checkpoints |
| Step convention and boundaries | p02, p15, p19; Session 1 Pitfall 2 |
| Activation values and properties | p02, p06, p11; Session 1 §3 |
| ReLU shapes and piecewise design | p03, p07, p18; Session 1 §4 |
| Forward passes and shapes | p04, p08; Session 2 §2–3 and Pitfall 4 |
| Region design (detectors, gates, polygons) | p09, p13, p14, p17; Session 2 §4–6 |
| Init variance derivation and simulations | p10, p12, p16; Session 3 §3–5 |

A final self-test worth doing cold: re-derive the $1/\sqrt{C}$ rule on
paper, naming the licensing fact at every step (the p12 rubric), and
hand-build the AND network for one fresh triangle-free polygon of your own
invention.

## Self-quiz answers

<details><summary><b>Open after committing to all 14 answers</b></summary>

**A1.** $z = 2 - 4 + 3 = 1 \ge 0$ → output 1.

**A2.** $x_1 - x_2 - 1 \ge 0$: $w = (1, -1)$, $b = -1$.

**A3.** $(0, 1, 1)$. The boundary $z = 0$ *fires* — thresholding is
$\ge$, always.

**A4.** They differ exactly on $z = 0$; the course pins `>=`. Designed
integer weights place test points *on* boundaries all the time, so the
strict version misclassifies every edge/corner probe.

**A5.** Step: $\{0, 1\}$; tanh: $(-1, 1)$; ReLU: $[0, \infty)$. Only
tanh is differentiable everywhere.

**A6.** The two affine maps compose into one affine map — the network
computes a single half-plane decision; the collapsed parameters are
$W^{\mathrm{eff}}$ and $b^{\mathrm{eff}}$.

**A7.** $r(-1) = 0$, $r(1) = 3$, $r(5) = 15 - 9 = 6$. Zero for
$z \le 0$, slope 3 on $[0, 2]$, constant 6 for $z \ge 2$.

**A8.** `np.max` aggregated along axis 0 instead of comparing
elementwise; use `np.maximum(z, 0.0)`.

**A9.** $W^{(1)}$: $(6, 4)$; $b^{(1)}$: $(6,)$; $W^{(2)}$: $(2, 6)$;
$b^{(2)}$: $(2,)$. Rows of $W$ enumerate units.

**A10.** $z^{(1)} = (1 - 1 + 0,\; 2 - 1) = (0, 1)$; $h = (1, 1)$
(boundary fires); $y = 2 - 1 - 0.5 = 0.5$.

**A11.** $8 - x_1 - 2x_2 \ge 0$: $w = (-1, -2)$, $b = 8$. $w$ is normal
to the line and points into the firing (inside) side.

**A12.** Weights $(1, 1, 1, 1)$, bias $-3.5$. On an edge the tight
detector reads $z = 0$ and fires by the $\ge$ convention, so all $m$
fire and the AND fires — edges and corners belong to the region.

**A13.** $\sigma = 1/\sqrt{49} = 1/7$. With $\sigma = 0.2$: factor
$\sigma\sqrt{C} = 0.2 \cdot 7 = 1.4$ — growing.

**A14.** $\operatorname{Var}[Wx] = E[W^2]E[x^2] = 0.09 \cdot 1 = 0.09$.
Facts: independence of $W^2$ and $x^2$ (functions of independent
variables) for the product rule, and the zero-mean shortcut
$\operatorname{Var} = E[\cdot^2]$ (twice).

</details>